In [11]:
import pandas as pd
import sqlite3
import os

#load in csvs and make database
cursor.execute("DROP TABLE IF EXISTS books")
print("Dropped old books table")

DATA_DIR = os.getcwd()
TMDB_CSV = os.path.join(DATA_DIR, "tmdb_movies_dataset.csv")
BOOKS_CSV = os.path.join(DATA_DIR, "goodreads.csv")
DB_PATH = os.path.join(DATA_DIR, "book_to_movie.db")

# CREATE TABLES

conn = sqlite3.connect(DB_PATH)
cursor = conn.cursor()

cursor.execute('''
CREATE TABLE IF NOT EXISTS movies (
    id INTEGER PRIMARY KEY,
    title TEXT,
    release_date TEXT,
    popularity REAL,
    vote_average REAL,
    vote_count INTEGER,
    original_language TEXT,
    overview TEXT,
    adult INTEGER,
    genre_ids TEXT
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS books (
    book_id INTEGER PRIMARY KEY AUTOINCREMENT,
    title TEXT,
    author TEXT,
    publisher TEXT,
    publication_date TEXT,
    genres TEXT,
    rating REAL,
    voters_count INTEGER,
    book_description TEXT,
    rating_distribution TEXT,
    into_movie INTEGER
)
''')

cursor.execute('''
CREATE TABLE IF NOT EXISTS book_movie_links (
    link_id INTEGER PRIMARY KEY AUTOINCREMENT,
    book_title TEXT,
    movie_title TEXT,
    match_type TEXT,
    confidence REAL
)
''')

conn.commit()
print("Tables: OK")

# ============================================================
# LOAD MOVIES
# ============================================================

try:
    movies_df = pd.read_csv(TMDB_CSV)
    print(f"Movies: {len(movies_df)}")
    
    movie_cols = ['id', 'title', 'release_date', 'popularity', 
                  'vote_average', 'vote_count', 'original_language',
                  'overview', 'adult', 'genre_ids']
    
    existing_cols = [col for col in movie_cols if col in movies_df.columns]
    movies_clean = movies_df[existing_cols].copy()
    movies_clean = movies_clean.where(pd.notnull(movies_clean), None)
    
    movies_clean.to_sql('movies', conn, if_exists='replace', index=False)
    print(f"Movies: {len(movies_clean)} inserted")
    
except FileNotFoundError:
    print("Movies: ERROR - file not found")

# ============================================================
# LOAD BOOKS
# ============================================================

try:
    books_df = pd.read_csv(BOOKS_CSV)
    print(f"Books: {len(books_df)}")
    
    book_cols_map = {
        'title': 'title',
        'author': 'author',
        'publisher': 'publisher',
        'publication_date': 'publication_date',
        'genres': 'genres',
        'rating': 'rating',
        'voters_count': 'voters_count',
        'book_description': 'book_description',
        'rating_distribution': 'rating_distribution',
        'into_movie': 'into_movie'
    }
    
    existing_cols = [col for col in book_cols_map.keys() if col in books_df.columns]
    
    if existing_cols:
        books_clean = books_df[existing_cols].copy()
        books_clean = books_clean.rename(columns=book_cols_map)
        
        if 'into_movie' in books_clean.columns:
            books_clean['into_movie'] = books_clean['into_movie'].astype(int)
        
        books_clean = books_clean.where(pd.notnull(books_clean), None)
        books_clean.to_sql('books', conn, if_exists='replace', index=False)
        print(f"Books: {len(books_clean)} inserted")
    else:
        print(f"Books: ERROR - no matching columns")
    
except FileNotFoundError:
    print("Books: ERROR - file not found")



Dropped old books table
Tables: OK
Movies: 4000
Movies: 4000 inserted
Books: 30005
Books: 30005 inserted


In [4]:
import pandas as pd

df = pd.read_csv("goodreads.csv")
print("Column names:")
print(df.columns.tolist())
print("\nFirst row:")
print(df.iloc[0].to_dict())

Column names:
['Unnamed: 0', 'title', 'author', 'author_url', 'rating', 'voters_count', 'url', 'into_movie', 'gender', 'hometown', 'books-num', 'born-at', 'works-count', 'publication-date', 'num-pages', 'popular-shelves', 'rating-dist', 'publisher', 'book-description']

First row:
{'Unnamed: 0': 0, 'title': 'The Sword and The Prophet (Syren, #1)', 'author': 'Missy LaRae', 'author_url': 'https://www.goodreads.com/author/show/5805389.Missy_LaRae', 'rating': 3.8, 'voters_count': 50, 'url': 'https://goodreads.com/book/show/13575987-the-sword-and-the-prophet', 'into_movie': 0, 'gender': 'female', 'hometown': 'Portland, Oregon', 'books-num': 2.0, 'born-at': '1979/12/17', 'works-count': 2.0, 'publication-date': '3/29/2012', 'num-pages': 150.0, 'popular-shelves': '[to-read, young-adult, fantasy, library, kindle, series, currently-reading, magic, ebooks, paranormal, digital-books, unknown-magic, where-do-i-own-if-not-maybe-buy, books-i-don-t-have-yet, 0-to-read-someday, books-on-my-kindle, e-bo